## Flan-T5 experiments with MLflow

Compare different Flan-T5 model sizes and generation configs; track runs and register the best model.

**No server required:** runs are saved to `./mlruns`. To view the UI later, run `mlflow ui -p 5000` in this folder.

In [1]:
# Run this cell once to install dependencies
#!pip install transformers datasets evaluate mlflow torch accelerate absl-py rouge-score nltk

In [4]:
#pip install azure-storage-blob

In [1]:
import warnings
warnings.filterwarnings("ignore")

import mlflow
import mlflow.transformers
from transformers import pipeline
from datasets import load_dataset
import evaluate

### Load evaluation data

#### Dataset link: https://huggingface.co/datasets/EdinburghNLP/xsum

Using a small subset of a summarization dataset to compare models.

In [2]:
# Small subset for quick evaluation (use more for serious comparison)
eval_data = load_dataset("xsum", split="test[:5]", trust_remote_code=True)
documents = eval_data["document"]
references = eval_data["summary"]
print(f"Loaded {len(documents)} examples for evaluation.")

# Print data sample (first 2 examples)
for i in range(min(2, len(documents))):
    print(f"\n--- Sample {i+1} ---")
    doc = documents[i]
    print(f"Document ({len(doc)} chars): {doc[:400]}..." if len(doc) > 400 else f"Document: {doc}")
    print(f"Reference summary: {references[i]}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'xsum' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loaded 5 examples for evaluation.

--- Sample 1 ---
Document (3525 chars): Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.
Workers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders.
The Welsh Government said more people than ever were getting help to address housing problems.
Changes to the Housing Act in Wales, introduced...
Reference summary: There is a "chronic" need for more housing for prison leavers in Wales, according to a charity.

--- Sample 2 ---
Document: Officers searched properties in the Waterfront Park and Colonsay View areas of the city on Wednesday.
Detectives said three firearms, ammunition and a five-figure sum of money were recovered.
A 26-year-old man who was arrested and charged appeared at Edinburgh Sheriff Court on Thursday.
Reference summary: A man has appeared in court after firearms, ammunition and cash were se

### Prompt types

Two prompting styles to compare per model (logged as `prompt_type` in MLflow for versioning).

In [3]:
# Two prompt types to compare (each run logs prompt_type in MLflow for versioning)
PROMPT_TYPES = {
    "instruction": "Summarize the following text:\n\n{doc}",
    "one_sentence": "Write a one-sentence summary of the following article:\n\n{doc}",
}

def get_model_size(pipe):
    """Return (num_parameters, size_mb) for the pipeline's model (float32)."""
    model = pipe.model
    n_params = sum(p.numel() for p in model.parameters())
    size_mb = n_params * 4 / (1024 ** 2)  # 4 bytes per float32 param
    return n_params, size_mb

def evaluate_summarization(pipe, documents, references, max_new_tokens=50, prompt_template=None):
    """Generate summaries using the given prompt template and compute ROUGE-1, ROUGE-2, ROUGE-L."""
    if prompt_template is None:
        prompt_template = PROMPT_TYPES["instruction"]
    rouge = evaluate.load("rouge")
    predictions = []
    for doc in documents:
        prompt = prompt_template.format(doc=doc)
        out = pipe(prompt, max_new_tokens=max_new_tokens, do_sample=False)
        pred = out[0]["generated_text"].strip() if out else ""
        predictions.append(pred)
    results = rouge.compute(predictions=predictions, references=references)
    return results, predictions

### Experiment configs

Define model variants and generation settings to compare.

In [4]:
# Each (model, prompt_type) = one run. Compare in MLflow by params: model_id, prompt_type.
experiments = [
    {"run_name": "Flan-T5-Small_instruction", "model_id": "google/flan-t5-small", "max_new_tokens": 50, "prompt_type": "instruction"},
    {"run_name": "Flan-T5-Small_one_sentence", "model_id": "google/flan-t5-small", "max_new_tokens": 50, "prompt_type": "one_sentence"},
    {"run_name": "Flan-T5-Base_instruction", "model_id": "google/flan-t5-base", "max_new_tokens": 50, "prompt_type": "instruction"},
    {"run_name": "Flan-T5-Base_one_sentence", "model_id": "google/flan-t5-base", "max_new_tokens": 50, "prompt_type": "one_sentence"},
]

### Run experiments and log to MLflow

Each run logs **model_id**, **prompt_type**, and **max_new_tokens**. In the MLflow UI, filter or sort by these params to compare versions (e.g. same model with different prompts).

In [5]:
# Uses local ./mlruns (no server needed). To view UI: run 'mlflow ui -p 5000' in this folder.
# Use a new name if the previous experiment was deleted (MLflow soft-deletes experiments).
mlflow.set_experiment("Flan-T5 Summarization Runs")

for config in experiments:
    run_name = config["run_name"]
    model_id = config["model_id"]
    max_new_tokens = config["max_new_tokens"]
    prompt_type = config["prompt_type"]
    prompt_template = PROMPT_TYPES[prompt_type]
    
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("model_id", model_id)
        mlflow.log_param("max_new_tokens", max_new_tokens)
        mlflow.log_param("prompt_type", prompt_type)
        
        pipe = pipeline(
            "text2text-generation",
            model=model_id,
            device=-1,
        )
        
        n_params, size_mb = get_model_size(pipe)
        print(f"{run_name}: {n_params/1e6:.2f}M parameters, ~{size_mb:.1f} MB")
        mlflow.log_metric("model_params_M", n_params / 1e6)
        mlflow.log_metric("model_size_mb", round(size_mb, 2))
        
        results, _ = evaluate_summarization(pipe, documents, references, max_new_tokens, prompt_template=prompt_template)
        
        mlflow.log_metric("rouge1", results["rouge1"])
        mlflow.log_metric("rouge2", results["rouge2"])
        mlflow.log_metric("rougeL", results["rougeL"])
        
        mlflow.transformers.log_model(
            transformers_model=pipe,
            name="model",
        )
        print(f"Logged {run_name}: ROUGE-L = {results['rougeL']:.4f}")

2026/03/10 20:30:52 INFO mlflow.tracking.fluent: Experiment with name 'Flan-T5 Summarization Runs' does not exist. Creating a new experiment.
Device set to use cpu


Flan-T5-Small_instruction: 76.96M parameters, ~293.6 MB


Token indices sequence length is longer than the specified maximum sequence length for this model (782 > 512). Running this sequence through the model will result in indexing errors
2026/03/10 20:30:57 WARNING mlflow.utils.requirements_utils: Found torch version (2.8.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.8.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/10 20:30:57 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.23.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.23.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/10 20:30:57 WARNING mlflow.utils.environment

Logged Flan-T5-Small_instruction: ROUGE-L = 0.3074


Device set to use cpu


...

Logged Flan-T5-Base_instruction: ROUGE-L = 0.2984


Device set to use cpu


Flan-T5-Base_one_sentence: 247.58M parameters, ~944.4 MB


Token indices sequence length is longer than the specified maximum sequence length for this model (788 > 512). Running this sequence through the model will result in indexing errors
2026/03/10 20:31:48 WARNING mlflow.utils.requirements_utils: Found torch version (2.8.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.8.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/10 20:31:48 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.23.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.23.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/10 20:31:48 WARNING mlflow.utils.environment

Logged Flan-T5-Base_one_sentence: ROUGE-L = 0.2880


### Register model (after choosing the best run)

1. Run `mlflow ui -p 5000` in this project folder, open http://localhost:5000, go to **Flan-T5 Summarization Runs**.
2. Compare runs by ROUGE metrics and pick the best **run_id**.
3. Set `run_id` and `model_name` below and run this cell.

In [6]:
# Uses same local ./mlruns as above (no server needed)

run_id = "9a786c43b26b4890b8c5fbf53c004ddb"  # e.g. from the MLflow UI
model_name = "Flan-T5-Summarization"  # name in the Model Registry

model_uri = f"runs:/{run_id}/model"
result = mlflow.register_model(model_uri=model_uri, name=model_name)
print(f"Registered '{result.name}' as version {result.version}")

Successfully registered model 'Flan-T5-Summarization'.
2026/03/10 20:40:04 WARNING mlflow.tracking._model_registry.fluent: Run with id 9a786c43b26b4890b8c5fbf53c004ddb has no artifacts at artifact path 'model', registering model based on models:/m-dd3beb01ddf54c3c8fd241c62f5decad instead


Registered 'Flan-T5-Summarization' as version 1


Created version '1' of model 'Flan-T5-Summarization'.


### Export registered model to folder

Download the champion (or specified) model to the **model** folder for Docker / local use.

In [9]:
import os

model_name = "Flan-T5-Summarization"
alias = "champion"  # or use version: client.get_model_version(name, version)
export_path = "model"

client = mlflow.MlflowClient()
mv = client.get_model_version_by_alias(model_name, alias)
artifact_uri = mv.source  # e.g. runs:/<run_id>/model

if os.path.exists(export_path):
    import shutil
    shutil.rmtree(export_path)
mlflow.artifacts.download_artifacts(artifact_uri=artifact_uri, dst_path=export_path)
print(f"Exported model (version {mv.version}) to folder: {export_path}/")

Exported model (version 1) to folder: model/


### Upload model to Azure Blob Storage

Export the model from MLflow (or use the existing `model/` folder from the cell above), zip it, and upload to an Azure Blob container. Use the blob URL (with SAS token for private containers) as `AZURE_MODEL_BLOB_URL` when running the summarizer app.

**Prerequisites:** `pip install azure-storage-blob` and Azure Storage account + container created. For private access, generate a SAS token with Read permission on the container or blob.

In [ ]:
# Option A: Upload from already-exported model/ folder (run "Export registered model to folder" cell first)
# Option B: Download from MLflow in this cell, then zip and upload

import os
import zipfile
from pathlib import Path
from azure.storage.blob import BlobServiceClient

# --- 1) Ensure we have the model on disk ---
model_name = "Flan-T5-Summarization"
alias = "champion"
export_path = "model"

# If model/ doesn't exist or you want to re-download from MLflow:
if not os.path.exists(export_path) or not (Path(export_path) / "MLmodel").exists():
    client = mlflow.MlflowClient()
    mv = client.get_model_version_by_alias(model_name, alias)
    artifact_uri = mv.source
    if os.path.exists(export_path):
        import shutil
        shutil.rmtree(export_path)
    mlflow.artifacts.download_artifacts(artifact_uri=artifact_uri, dst_path=export_path)
    print(f"Downloaded model (v{mv.version}) to {export_path}/")
else:
    print(f"Using existing {export_path}/")

# --- 2) Zip the model folder (zip contents = model files at root, for app's extract logic) ---
zip_path = "model.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(export_path):
        for f in files:
            path = os.path.join(root, f)
            arcname = os.path.relpath(path, export_path)
            zf.write(path, arcname)
print(f"Created {zip_path} ({os.path.getsize(zip_path) / (1024*1024):.1f} MB)")

# --- 3) Upload to Azure Blob Storage ---

# Load .env 
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

AZURE_STORAGE_CONNECTION_STRING = os.environ.get("AZURE_STORAGE_CONNECTION_STRING", "")
CONTAINER_NAME = os.environ.get("AZURE_MODEL_CONTAINER", "models")  # container name
BLOB_NAME = "model.zip"  # blob path inside the container

if not AZURE_STORAGE_CONNECTION_STRING or AZURE_STORAGE_CONNECTION_STRING.startswith("<"):
    print("Set AZURE_STORAGE_CONNECTION_STRING (and optionally AZURE_MODEL_CONTAINER) then re-run this cell.")
else:
    blob_service = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)
    container_client = blob_service.get_container_client(CONTAINER_NAME)
    # Create container if it doesn't exist (avoids ContainerNotFound)
    if not container_client.exists():
        container_client.create_container()
        print(f"Created container '{CONTAINER_NAME}'.")
    blob_client = container_client.get_blob_client(BLOB_NAME)
    with open(zip_path, "rb") as f:
        blob_client.upload_blob(f, overwrite=True)
    # Build the URL (for private container, add ?sas_token or use blob_client.url with SAS)
    account_name = blob_client.account_name
    url = f"https://{account_name}.blob.core.windows.net/{CONTAINER_NAME}/{BLOB_NAME}"
    print(f"Uploaded to Azure Blob: {url}")
    print("For private container: generate a SAS token (Read) in Azure Portal and use:")
    print(f"  AZURE_MODEL_BLOB_URL = \"{url}?<your-sas-token>\"")

Using existing model/
Created model.zip (273.0 MB)
Created container 'models'.
Uploaded to Azure Blob: https://storagemodelllmops.blob.core.windows.net/models/model.zip
For private container: generate a SAS token (Read) in Azure Portal and use:
  AZURE_MODEL_BLOB_URL = "https://storagemodelllmops.blob.core.windows.net/models/model.zip?<your-sas-token>"


### Load a registered model for inference

In [8]:
# Load by model alias (champion = best model version in registry)
pipe = mlflow.transformers.load_model("models:/Flan-T5-Summarization@champion")
pipe("Summarize: Machine learning is a subset of artificial intelligence.", max_new_tokens=30)

# Or by stage: models:/Flan-T5-Summarization/Production
# Or by run_id: runs:/<run_id>/model

2026/03/10 20:58:23 INFO mlflow.transformers: 'models:/Flan-T5-Summarization@champion' resolved as 'file:d:/Sajjad-Workspace/MLOps/mlruns/1/models/m-dd3beb01ddf54c3c8fd241c62f5decad/artifacts'
`torch_dtype` is deprecated! Use `dtype` instead!
2026/03/10 20:58:26 WARNING mlflow.transformers.model_io: Could not specify device parameter for this pipeline type.Falling back to loading the model with the default device.
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


[{'generated_text': 'Artificial intelligence is a subset of artificial intelligence.'}]